# 04 — TrustMind RAG Evaluation (LLM + Hybrid Retrieval)

Second experimental arm for the dissertation RQ:

> To what extent does RAG improve trustworthiness, reliability and explainability of LLM-generated wellbeing assessments compared with a standalone LLM?

**Fair comparison controls (match `02_LLM_Baseline.ipynb`):**
- Model: `gpt-4.1`
- Sample size: 100
- Seed: 42
- Test set: `datasets/synthetic_wellbeing/test.csv`
- Temperature: 0.0

Does **not** modify the LLM-only baseline.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if not (ROOT / "rag").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "research"))

load_dotenv(ROOT / "research" / ".env")
load_dotenv(ROOT / ".env")

from llm_baseline import VALID_LABELS, compute_metrics, load_and_sample_test
from rag.config import get_rag_config
from rag.rag_pipeline import RagPipeline

cfg = get_rag_config()
RESULTS = cfg.results_dir
FIGURES = ROOT / "research" / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

DRY_RUN = False  # True = sample only, no API calls
print("model=", cfg.gpt_model, "top_k=", cfg.top_k, "sample=", cfg.sample_size)

## Prerequisites

Run once from repo root before evaluating:

```bash
pip install -r requirements-rag.txt
python scripts/chunk_documents.py --force
python scripts/generate_embeddings.py
python scripts/build_faiss.py
python scripts/build_bm25.py
```

In [ ]:
sample = load_and_sample_test(cfg.test_csv, cfg.sample_size, cfg.random_seed)
print(sample["true_label"].value_counts())
sample.head(3)

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True — skipping RAG inference")
    preds = pd.DataFrame()
else:
    if not cfg.openai_api_key:
        raise RuntimeError("Set OPENAI_API_KEY in research/.env")
    client = OpenAI(api_key=cfg.openai_api_key)
    pipeline = RagPipeline(cfg, client=client)
    rows = []
    for i, row in sample.iterrows():
        out = pipeline.run(str(row["text"]))
        rows.append(
            {
                "text": row["text"],
                "true_label": row["true_label"],
                "predicted_label": out["predicted_label"],
                "confidence": out["confidence"],
                "reasoning": out["reasoning"],
                "retrieved_sources": json.dumps(out["retrieved_sources"]),
                "n_retrieved": len(out.get("retrieved_passages") or []),
                "latency_ms": out["latency_ms"],
                "parse_ok": out["parse_ok"],
                "error": out["error"],
            }
        )
        if (int(i) + 1) % 10 == 0:
            print(f"Processed {int(i)+1}/{len(sample)}")
        if cfg.sleep_between_calls > 0:
            import time

            time.sleep(cfg.sleep_between_calls)
    preds = pd.DataFrame(rows)
    out_csv = RESULTS / "rag_predictions.csv"
    preds.to_csv(out_csv, index=False)
    print("Saved", out_csv)

In [ ]:
if not preds.empty:
    metrics = compute_metrics(
        preds["true_label"].tolist(),
        preds["predicted_label"].tolist(),
    )
    payload = {
        "experiment": "llm_rag_hybrid",
        "model_name": cfg.gpt_model,
        "embedding_model": cfg.embedding_model,
        "sample_size": cfg.sample_size,
        "random_seed": cfg.random_seed,
        "temperature": cfg.temperature,
        "top_k": cfg.top_k,
        "chunk_size_words": cfg.chunk_size_words,
        "chunk_overlap_words": cfg.chunk_overlap_words,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": metrics,
        "mean_confidence": float(preds["confidence"].mean()),
        "mean_n_retrieved": float(preds["n_retrieved"].mean()),
        "mean_latency_ms": float(preds["latency_ms"].mean()),
    }
    metrics_path = RESULTS / "rag_metrics.json"
    metrics_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(metrics["classification_report_text"])
    print("accuracy", metrics["accuracy"], "macro_f1", metrics["f1_macro"])
else:
    metrics = {}

In [ ]:
if metrics:
    cm = np.array(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(VALID_LABELS)))
    ax.set_yticks(range(len(VALID_LABELS)))
    ax.set_xticklabels(VALID_LABELS, rotation=45, ha="right")
    ax.set_yticklabels(VALID_LABELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("RAG confusion matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(FIGURES / "rag_confusion_matrix.png", dpi=150)
    plt.show()